<a href="https://colab.research.google.com/github/Hamsini11/ADIP/blob/main/Extern_MortgageDocumentIntelligenceCapstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone Project - AI/ML Externship Program

In [ ]:
!pip install -q sentence-transformers faiss-cpu PyPDF2 gradio pdf2image pytesseract pillow
!apt-get install -y poppler-utils tesseract-ocr > /dev/null 2>&1
print("✅ Installed with OCR support")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 13.2 MB/s eta 0:00:00
✅ Installed with OCR support


In [ ]:
import PyPDF2
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import gradio as gr
import re
from datetime import datetime

print("✅ Imports complete")

✅ Imports complete


In [ ]:
# FHA Rules and Mortgage Knowledge (built-in)
FHA_KNOWLEDGE = {
    "fha_requirements": """
FHA LOAN REQUIREMENTS:
- Minimum Credit Score: 500 (with 10% down) or 580 (with 3.5% down)
- Debt-to-Income Ratio: Maximum 43% (up to 50% with compensating factors)
- Down Payment: Minimum 3.5% for credit score 580+
- Property Type: Primary residence only (1-4 unit properties)
- Mortgage Insurance: Required for life of loan if down payment < 10%
- Employment: 2 years steady employment history required
- Income Documentation: 2 years tax returns, 2 months paystubs, W2 forms
- Assets: 2 months bank statements showing reserves
- Property Standards: Must meet HUD minimum property standards
- Loan Limits: Vary by county (2025: $498,257 for single-family in most areas)
""",

    "required_documents": """
REQUIRED DOCUMENTS FOR FHA LOAN APPROVAL:
✓ Loan Application (Form 1003)
✓ Government-issued ID (driver's license, passport)
✓ Social Security Card or proof of SSN
✓ Pay stubs (most recent 2 months)
✓ W2 forms (past 2 years)
✓ Federal tax returns (past 2 years) with all schedules
✓ Bank statements (all accounts, past 2 months)
✓ Credit report (lender will pull)
✓ Employment verification letter
✓ Property appraisal (ordered by lender)
✓ Homeowners insurance quote
✓ Purchase agreement (for home purchases)
✓ FHA case number assignment
""",

    "conventional_requirements": """
CONVENTIONAL LOAN REQUIREMENTS:
- Minimum Credit Score: 620 (higher scores get better rates)
- Debt-to-Income Ratio: Maximum 43% (up to 50% with strong compensating factors)
- Down Payment: Minimum 3% (5% for investment properties)
- PMI Required: If down payment < 20%
- PMI Removal: Automatic at 78% LTV, can request at 80% LTV
- Employment: 2 years steady employment
- Reserves: 2-6 months PITI required
- Property Standards: Must meet Fannie Mae/Freddie Mac standards
- Loan Limits: $766,550 for single-family (2025 conforming limit)
""",

    "mortgage_insurance": """
MORTGAGE INSURANCE EXPLAINED:
- FHA MIP: Upfront (1.75% of loan) + Annual (0.55-0.85% depending on LTV)
- FHA MIP Duration: Life of loan if down payment < 10%, or 11 years if 10%+
- Conventional PMI: 0.3-1.5% of loan amount annually (based on credit/LTV)
- PMI Cancellation: Automatic at 78% LTV, borrower can request at 80% LTV
- VA Funding Fee: 2.15-3.3% upfront (no monthly premium)
""",

    "dti_calculation": """
DEBT-TO-INCOME RATIO CALCULATION:
Front-end DTI = (Housing expenses / Gross monthly income) × 100
Back-end DTI = (All monthly debts / Gross monthly income) × 100

Housing expenses include: PITI (Principal, Interest, Taxes, Insurance) + HOA

FHA Guidelines:
- Front-end: Max 31%
- Back-end: Max 43% (up to 50% with compensating factors)

Conventional Guidelines:
- Front-end: Max 28%
- Back-end: Max 43%
"""
}

print("✅ FHA knowledge base loaded")

✅ FHA knowledge base loaded


In [ ]:
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import os

# Global storage
faiss_index = None
chunk_store = []
embedding_model = None
document_metadata = {}

def extract_text_with_ocr(pdf_path):
    """
    Extract text using OCR (handles images and poorly formatted PDFs)
    """
    try:
        # Try PyPDF2 first (faster)
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)

            all_pages_text = []
            ocr_needed = False

            for page_num, page in enumerate(pdf_reader.pages):
                text = page.extract_text()

                # If page has very little text, use OCR
                if len(text.strip()) < 100:
                    ocr_needed = True
                    print(f"⚠️ Page {page_num+1} has minimal text, switching to OCR...")
                    break

                all_pages_text.append({
                    'page_num': page_num + 1,
                    'text': text,
                    'method': 'pypdf2'
                })

            # If OCR not needed, return PyPDF2 results
            if not ocr_needed:
                return all_pages_text

    except Exception as e:
        print(f"⚠️ PyPDF2 extraction failed: {e}")
        ocr_needed = True

    # Use OCR if PyPDF2 failed or text too short
    print("🔄 Using OCR for text extraction...")

    try:
        # Convert PDF to images
        images = convert_from_path(pdf_path, dpi=300)

        all_pages_text = []
        for i, image in enumerate(images):
            print(f"📸 OCR on page {i+1}...")
            text = pytesseract.image_to_string(image)

            all_pages_text.append({
                'page_num': i + 1,
                'text': text,
                'method': 'ocr'
            })

        return all_pages_text

    except Exception as e:
        print(f"❌ OCR failed: {e}")
        return []

def classify_document(text):
    """Classify mortgage document type"""
    t = text.lower()

    # Comprehensive classification
    if any(x in t for x in ["uniform residential loan application", "form 1003", "section 1: borrower", "section 2: financial", "section 3: property"]):
        return "Loan Application (Form 1003)"

    elif "closing disclosure" in t:
        return "Closing Disclosure"

    elif any(x in t for x in ["loan estimate", "fee details", "your actual rate"]):
        return "Loan Estimate"

    elif any(x in t for x in ["earnings statement", "payslip", "pay date", "employee name", "employee id", "net pay", "total earnings", "deductions"]):
        return "Paystub"

    elif any(x in t for x in ["form w-2", "form w2", "wage and tax statement", "box 1", "box 2", "social security wages"]):
        return "W2 Tax Form"

    elif "form 1040" in t or "u.s. individual income tax return" in t:
        return "Tax Return (Form 1040)"

    elif any(x in t for x in ["bank statement", "account statement", "monthly statement", "beginning balance", "ending balance", "deposits and credits"]) or ("chase" in t and "account" in t):
        return "Bank Statement"

    elif any(x in t for x in ["appraisal report", "residential appraisal", "appraised value", "property information", "valuation analysis", "final appraised value"]):
        return "Property Appraisal"

    elif any(x in t for x in ["homeowners insurance", "insurance quote", "policy information", "coverage details", "dwelling coverage", "premium summary"]):
        return "Homeowners Insurance"

    elif any(x in t for x in ["employment contract", "employment agreement", "contract of employment"]):
        return "Employment Document"

    elif "credit report" in t or "fico" in t or "credit score" in t:
        return "Credit Report"

    else:
        return "Other Document"

def split_pdf_by_document(pdf_path):
    """Split PDF into documents using OCR-capable extraction"""

    # Extract text from all pages
    pages_data = extract_text_with_ocr(pdf_path)

    if not pages_data:
        return []

    documents = []
    current_doc = {"pages": [], "text": "", "type": None}

    for page_data in pages_data:
        page_text = page_data['text']
        page_num = page_data['page_num']

        # Classify document type
        doc_type = classify_document(page_text)

        # Debug output
        print(f"📄 Page {page_num}: {doc_type} ({len(page_text)} chars)")

        # New document detected
        if doc_type != current_doc["type"] and current_doc["text"]:
            documents.append(current_doc)
            current_doc = {"pages": [], "text": "", "type": doc_type}

        current_doc["pages"].append(page_num)
        current_doc["text"] += page_text + "\n"
        current_doc["type"] = doc_type

    # Add last document
    if current_doc["text"]:
        documents.append(current_doc)

    # Summary
    print(f"\n✅ EXTRACTED {len(documents)} DOCUMENTS:")
    for i, doc in enumerate(documents, 1):
        print(f"  {i}. {doc['type']} (Pages {doc['pages']}, {len(doc['text'])} chars)")

    return documents

print("✅ OCR-enabled document processing ready")

✅ OCR-enabled document processing ready


In [ ]:
def create_contextual_chunks(documents, chunk_size=500):
    """
    CONTEXTUAL RETRIEVAL IMPLEMENTATION
    Adds document metadata context to each chunk
    """
    contextual_chunks = []

    for doc in documents:
        # Create document context header
        doc_context = f"""Document Type: {doc['type']}
Pages: {doc['pages']}
Total Pages: {len(doc['pages'])}
Document Purpose: Mortgage loan package component

"""

        # Split into chunks
        words = doc['text'].split()

        for i in range(0, len(words), chunk_size):
            chunk_text = ' '.join(words[i:i+chunk_size])

            # Add context to chunk (CONTEXTUAL RETRIEVAL)
            contextualized_chunk = doc_context + chunk_text

            contextual_chunks.append({
                'original': chunk_text,
                'contextualized': contextualized_chunk,
                'doc_type': doc['type'],
                'pages': doc['pages'],
                'full_doc_text': doc['text']  # Keep full doc for precise extraction
            })

    return contextual_chunks

print("✅ Contextual chunking ready")

✅ Contextual chunking ready


In [ ]:
def build_vector_index(contextual_chunks):
    """Build FAISS vector index from contextual chunks"""
    global faiss_index, chunk_store, embedding_model

    print("🔄 Loading embedding model...")
    embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

    print("🔄 Generating embeddings...")
    # Embed the CONTEXTUALIZED chunks (not originals)
    texts = [chunk['contextualized'] for chunk in contextual_chunks]
    embeddings = embedding_model.encode(texts, show_progress_bar=False)

    print("🔄 Building FAISS index...")
    dimension = embeddings.shape[1]
    faiss_index = faiss.IndexFlatL2(dimension)
    faiss_index.add(np.array(embeddings).astype('float32'))

    chunk_store = contextual_chunks

    print(f"✅ Vector index built: {faiss_index.ntotal} chunks")
    return len(chunk_store)

print("✅ Vector indexing ready")

✅ Vector indexing ready


In [ ]:
def route_query(query):
    """
    MORAG ADAPTIVE ROUTING - Fixed priority order
    """
    q = query.lower()

    # PRIORITY 1: DENSE - FHA knowledge base queries (check FIRST)
    knowledge_keywords = ['fha requirement', 'fha rule', 'what are fha',
                         'what does fha', 'conventional requirement',
                         'what is dti', 'debt to income', 'what is pmi',
                         'mortgage insurance explain', 'fha guideline',
                         'what are the requirement', 'loan requirement']
    if any(kw in q for kw in knowledge_keywords):
        return "DENSE"

    # PRIORITY 2: FILTERED - Specific document type mentioned
    doc_keywords = ['paystub', 'w2', 'w-2', 'tax return', 'bank statement',
                   'appraisal', 'insurance', 'closing disclosure',
                   'loan application', 'in the paystub', 'from the w2',
                   'from paystub', 'show me paystub']
    if any(kw in q for kw in doc_keywords):
        return "FILTERED"

    # PRIORITY 3: HYBRID - Multi-step reasoning
    hybrid_keywords = ['qualified', 'eligible', 'dti', 'debt-to-income',
                      'can they afford', 'do they qualify', 'compare',
                      'ratio', 'afford', 'approval']
    if any(kw in q for kw in hybrid_keywords):
        return "HYBRID"

    # PRIORITY 4: SPARSE - ONLY exact extraction (most specific triggers)
    # Be very specific - only trigger on clear extraction queries
    if any(kw in q for kw in ['what is the loan amount', 'what is the interest rate',
                               'what is the borrower name', 'loan amount and',
                               'interest rate and', 'extract the']):
        return "SPARSE"

    # DEFAULT: DENSE for general questions
    return "DENSE"

print("✅ MoRAG router ready")

✅ MoRAG router ready


In [ ]:
def retrieve_relevant_chunks(query, k=5):
    """Retrieve top-k relevant chunks using vector similarity"""
    global faiss_index, chunk_store, embedding_model

    if faiss_index is None or not chunk_store:
        return []

    # Generate query embedding
    query_embedding = embedding_model.encode([query])

    # Search FAISS index
    distances, indices = faiss_index.search(
        np.array(query_embedding).astype('float32'), k
    )

    # Return retrieved chunks
    results = []
    for idx in indices[0]:
        if idx < len(chunk_store):
            results.append(chunk_store[idx])

    return results

print("✅ Retrieval function ready")

✅ Retrieval function ready


In [ ]:
def extract_precise_data(query, chunks, query_type):
    """
    Precise data extraction from retrieved chunks
    Fixes the SPARSE strategy accuracy issue
    """
    q_lower = query.lower()

    # Combine all retrieved text
    combined_text = ""
    full_docs = []
    for chunk in chunks:
        combined_text += chunk['original'] + " "
        if chunk['full_doc_text'] not in full_docs:
            full_docs.append(chunk['full_doc_text'])

    # For critical extractions, search FULL documents (not just chunks)
    search_text = " ".join(full_docs) if full_docs else combined_text

    # BORROWER NAME
    if any(kw in q_lower for kw in ['borrower', 'name', 'applicant', 'who is']):
        # Search patterns
        patterns = [
            r'Borrower Name:\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)',
            r'Employee Name:\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)',
            r'Account Holder:\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)',
            r'Insured:\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)',
            r'Borrower:\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+(?:\s*&\s*[A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)?)'
        ]

        for pattern in patterns:
            match = re.search(pattern, search_text)
            if match:
                name = match.group(1).strip()

                # Look for co-borrower
                coborrower_match = re.search(r'Co-Borrower[:\s]+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)', search_text)

                if coborrower_match:
                    coborrower = coborrower_match.group(1).strip()

                    # Find SSNs if available
                    ssn_pattern = r'SSN[:\s]+(XXX-XX-\d{4}|XXX-\d{2}-\d{4})'
                    ssns = re.findall(ssn_pattern, search_text)

                    # Find address
                    addr_match = re.search(r'(?:Current Address|Property Address|Address)[:\s]+([0-9]+\s+[A-Za-z\s]+,\s*[A-Za-z\s]+,\s*[A-Z]{2}\s+\d{5})', search_text)

                    result = f"📋 BORROWER INFORMATION:\n\n"
                    result += f"Primary Borrower: {name}\n"
                    if ssns and len(ssns) > 0:
                        result += f"  SSN: {ssns[0]}\n"
                    result += f"\nCo-Borrower: {coborrower}\n"
                    if ssns and len(ssns) > 1:
                        result += f"  SSN: {ssns[1]}\n"
                    if addr_match:
                        result += f"\nCurrent Address: {addr_match.group(1)}\n"

                    return result
                else:
                    # Single borrower
                    ssn_match = re.search(r'SSN[:\s]+(XXX-XX-\d{4})', search_text)
                    addr_match = re.search(r'(?:Current Address|Address)[:\s]+([0-9]+\s+[A-Za-z\s]+,\s*[A-Za-z\s]+,\s*[A-Z]{2}\s+\d{5})', search_text)

                    result = f"📋 BORROWER INFORMATION:\n\n"
                    result += f"Borrower: {name}\n"
                    if ssn_match:
                        result += f"SSN: {ssn_match.group(1)}\n"
                    if addr_match:
                        result += f"Address: {addr_match.group(1)}\n"

                    return result

        return "Borrower information found in documents. Primary borrower name detected in loan application."

    # LOAN AMOUNT
    if any(kw in q_lower for kw in ['loan amount', 'how much loan', 'principal']):
        patterns = [
            r'Loan Amount[:\s]+\$?([\d,]+\.?\d*)',
            r'Loan Amount Requested[:\s]+\$?([\d,]+\.?\d*)',
            r'Total Loan Amount[:\s]+\$?([\d,]+\.?\d*)'
        ]

        for pattern in patterns:
            match = re.search(pattern, search_text)
            if match:
                amount = match.group(1)

                # Look for additional details
                rate_match = re.search(r'Interest Rate[:\s]+([\d.]+)%', search_text)
                term_match = re.search(r'(\d+)[\s-]*Year', search_text, re.IGNORECASE)
                type_match = re.search(r'Loan Type[:\s]+([^\n]+)', search_text)

                result = f"💰 LOAN DETAILS:\n\n"
                result += f"Loan Amount: ${amount}\n"
                if rate_match:
                    result += f"Interest Rate: {rate_match.group(1)}%\n"
                if term_match:
                    result += f"Loan Term: {term_match.group(1)} years\n"
                if type_match:
                    loan_type = type_match.group(1).strip()
                    result += f"Loan Type: {loan_type}\n"

                return result

        return "Loan amount information found in Loan Application or Closing Disclosure."

    # INTEREST RATE
    if any(kw in q_lower for kw in ['interest rate', 'rate', 'apr']):
        rate_match = re.search(r'Interest Rate[:\s]+([\d.]+)\s*%', search_text)
        if rate_match:
            rate = rate_match.group(1)

            # Look for monthly payment
            payment_match = re.search(r'(?:Monthly Payment|Principal\s*&\s*Interest)[:\s]+\$?([\d,]+\.?\d*)', search_text)

            result = f"💳 INTEREST & PAYMENT:\n\n"
            result += f"Interest Rate: {rate}%\n"
            if payment_match:
                result += f"Monthly Principal & Interest: ${payment_match.group(1)}\n"

            return result

        return "Interest rate information in Closing Disclosure or Loan Estimate."

    # MONTHLY PAYMENT
    if any(kw in q_lower for kw in ['monthly payment', 'payment', 'piti']):
        patterns = [
            r'Total Estimated Monthly Payment[:\s]+\$?([\d,]+\.?\d*)',
            r'Monthly Payment[:\s]+\$?([\d,]+\.?\d*)',
            r'Principal\s*&\s*Interest[:\s]+\$?([\d,]+\.?\d*)'
        ]

        for pattern in patterns:
            match = re.search(pattern, search_text)
            if match:
                payment = match.group(1)

                # Break down components
                result = f"💵 MONTHLY PAYMENT BREAKDOWN:\n\n"

                pi_match = re.search(r'Principal\s*&\s*Interest[:\s]+\$?([\d,]+\.?\d*)', search_text)
                tax_match = re.search(r'Property Taxes[:\s]+\$?([\d,]+\.?\d*)', search_text)
                ins_match = re.search(r'Homeowner(?:\'s)?\s+Insurance[:\s]+\$?([\d,]+\.?\d*)', search_text)
                mi_match = re.search(r'Mortgage Insurance[:\s]+\$?([\d,]+\.?\d*)', search_text)

                if pi_match:
                    result += f"Principal & Interest: ${pi_match.group(1)}\n"
                if mi_match:
                    result += f"Mortgage Insurance: ${mi_match.group(1)}\n"
                if tax_match:
                    result += f"Property Taxes: ${tax_match.group(1)}\n"
                if ins_match:
                    result += f"Homeowners Insurance: ${ins_match.group(1)}\n"

                result += f"\nTotal Monthly Payment: ${payment}\n"

                return result

        return "Monthly payment information in Closing Disclosure."

    # INCOME
    if any(kw in q_lower for kw in ['income', 'salary', 'earnings', 'wages']):
        # Look for multiple income indicators
        monthly_match = re.search(r'(?:Monthly Income|Basic Pay)[:\s]+\$?([\d,]+\.?\d*)', search_text)
        annual_match = re.search(r'(?:Wages|Annual|Total Earnings)[:\s]+\$?([\d,]+\.?\d*)', search_text)
        ytd_match = re.search(r'YTD[:\s]+\$?([\d,]+\.?\d*)', search_text)

        if monthly_match or annual_match:
            result = f"💼 INCOME INFORMATION:\n\n"

            if monthly_match:
                result += f"Monthly Income: ${monthly_match.group(1)}\n"
            if annual_match:
                result += f"Annual Wages: ${annual_match.group(1)}\n"
            if ytd_match:
                result += f"Year-to-Date: ${ytd_match.group(1)}\n"

            # Look for employer
            emp_match = re.search(r'(?:Employer|Company)[:\s]+([A-Za-z\s&.]+?)(?:\n|,|\||$)', search_text)
            if emp_match:
                result += f"\nEmployer: {emp_match.group(1).strip()}\n"

            return result

        return "Income information found in paystubs and W2 forms."

    # PROPERTY ADDRESS/VALUE
    if any(kw in q_lower for kw in ['property', 'address', 'house', 'home']):
        addr_match = re.search(r'Property Address[:\s]+([0-9]+\s+[A-Za-z\s]+,\s*[A-Za-z\s]+,\s*[A-Z]{2}\s+\d{5})', search_text)
        value_match = re.search(r'(?:Appraised Value|Purchase Price|Sale Price)[:\s]+\$?([\d,]+\.?\d*)', search_text)

        if addr_match or value_match:
            result = f"🏠 PROPERTY INFORMATION:\n\n"

            if addr_match:
                result += f"Property Address: {addr_match.group(1)}\n"
            if value_match:
                result += f"Appraised Value: ${value_match.group(1)}\n"

            # Look for property details
            type_match = re.search(r'Property Type[:\s]+([^\n]+)', search_text)
            sqft_match = re.search(r'([\d,]+)\s+square feet', search_text, re.IGNORECASE)

            if type_match:
                result += f"Property Type: {type_match.group(1).strip()}\n"
            if sqft_match:
                result += f"Square Footage: {sqft_match.group(1)} sq ft\n"

            return result

        return "Property information in Loan Application and Appraisal Report."

    # CLOSING COSTS / FEES
    if any(kw in q_lower for kw in ['closing cost', 'fees', 'charges', 'expenses']):
        # Find total closing costs
        total_match = re.search(r'(?:TOTAL CLOSING COSTS?|Total Estimated)[:\s]+\$?([\d,]+\.?\d*)', search_text)

        if total_match:
            result = f"💵 CLOSING COSTS:\n\n"
            result += f"Total Closing Costs: ${total_match.group(1)}\n\n"

            # Extract individual fees
            result += "Major Fees:\n"

            fee_patterns = [
                (r'Origination[^$]*\$?([\d,]+\.?\d*)', 'Loan Origination'),
                (r'Underwriting Fee[:\s]+\$?([\d,]+\.?\d*)', 'Underwriting'),
                (r'Appraisal Fee[:\s]+\$?([\d,]+\.?\d*)', 'Appraisal'),
                (r'Title[^$]*Settlement[^$]*\$?([\d,]+\.?\d*)', 'Title/Settlement'),
                (r'Insurance[^$]*\$?([\d,]+\.?\d*)', 'Insurance')
            ]

            for pattern, name in fee_patterns:
                match = re.search(pattern, search_text, re.IGNORECASE)
                if match:
                    result += f"  • {name}: ${match.group(1)}\n"

            return result

        return "Closing costs detailed in Closing Disclosure."

    # EMPLOYMENT
    if any(kw in q_lower for kw in ['employment', 'employer', 'job', 'work']):
        emp_match = re.search(r'(?:Employer|Company)[:\s]+([A-Za-z\s&.,]+?)(?:\n|Employee|\||$)', search_text)
        pos_match = re.search(r'(?:Position|Department)[:\s]+([A-Za-z\s]+?)(?:\n|Pay|\||$)', search_text)

        if emp_match:
            result = f"💼 EMPLOYMENT INFORMATION:\n\n"
            result += f"Employer: {emp_match.group(1).strip()}\n"

            if pos_match:
                result += f"Position: {pos_match.group(1).strip()}\n"

            # Look for employment dates
            date_match = re.search(r'Pay Date[:\s]+(\d{2}/\d{2}/\d{4})', search_text)
            if date_match:
                result += f"Recent Pay Date: {date_match.group(1)}\n"

            return result

        return "Employment information in paystubs and employment verification documents."

    # Default: return first chunk content
    if chunks:
        return f"📄 INFORMATION FROM {chunks[0]['doc_type']}:\n\n{chunks[0]['original'][:400]}..."

    return "Information not found in uploaded documents."

print("✅ Smart extraction logic ready")

✅ Smart extraction logic ready


In [ ]:
def check_fha_completeness(documents):
    """
    Check FHA loan package completeness
    Returns what's present and what's missing
    """
    doc_types = [doc['type'] for doc in documents]

    # Define requirements
    requirements = {
        "Loan Application (Form 1003)": False,
        "Government ID": False,
        "Paystubs (2 months)": False,
        "W2 Forms (2 years)": False,
        "Tax Returns (2 years)": False,
        "Bank Statements (2 months)": False,
        "Credit Report": False,
        "Employment Verification": False,
        "Property Appraisal": False,
        "Homeowners Insurance": False,
        "FHA Case Number": False
    }

    # Check what's present
    for doc_type in doc_types:
        if "Loan Application" in doc_type or "Form 1003" in doc_type:
            requirements["Loan Application (Form 1003)"] = True

        if "Paystub" in doc_type:
            # Count paystubs
            paystub_count = sum(1 for dt in doc_types if "Paystub" in dt)
            if paystub_count >= 2:
                requirements["Paystubs (2 months)"] = True

        if "W2" in doc_type or "W-2" in doc_type:
            requirements["W2 Forms (2 years)"] = True

        if "Tax Return" in doc_type or "1040" in doc_type:
            requirements["Tax Returns (2 years)"] = True

        if "Bank Statement" in doc_type:
            bank_count = sum(1 for dt in doc_types if "Bank Statement" in dt)
            if bank_count >= 2:
                requirements["Bank Statements (2 months)"] = True

        if "Credit Report" in doc_type:
            requirements["Credit Report"] = True

        if "Employment" in doc_type:
            requirements["Employment Verification"] = True

        if "Appraisal" in doc_type:
            requirements["Property Appraisal"] = True

        if "Insurance" in doc_type and "Homeowner" in doc_type:
            requirements["Homeowners Insurance"] = True

    # Separate present and missing
    present = [req for req, status in requirements.items() if status]
    missing = [req for req, status in requirements.items() if not status]

    return present, missing

print("✅ Completeness checker ready")

✅ Completeness checker ready


In [ ]:
def handle_query(query, documents, chunks):
    """
    Main query handling with routing and response generation
    """
    q_lower = query.lower()

    # Check if asking about FHA rules (not in uploaded doc)
    fha_rule_keywords = ['fha requirement', 'fha rule', 'what does fha',
                         'fha guideline', 'fha minimum', 'fha allow',
                         'conventional requirement', 'conventional vs fha',
                         'what is dti', 'debt to income', 'what is pmi',
                         'mortgage insurance explain']

    if any(kw in q_lower for kw in fha_rule_keywords):
        # Answer from FHA knowledge base
        if 'fha requirement' in q_lower or 'fha guideline' in q_lower or 'fha minimum' in q_lower:
            return f"📚 FHA LOAN REQUIREMENTS:\n\n{FHA_KNOWLEDGE['fha_requirements']}\n\n" + \
                   f"Source: FHA/HUD Guidelines (Built-in Knowledge)"

        elif 'conventional' in q_lower:
            return f"📚 CONVENTIONAL LOAN REQUIREMENTS:\n\n{FHA_KNOWLEDGE['conventional_requirements']}\n\n" + \
                   f"For comparison with FHA requirements, see above.\n\nSource: Fannie Mae/Freddie Mac Guidelines"

        elif 'dti' in q_lower or 'debt to income' in q_lower or 'debt-to-income' in q_lower:
            return f"📊 DEBT-TO-INCOME RATIO:\n\n{FHA_KNOWLEDGE['dti_calculation']}\n\n" + \
                   f"Source: FHA/Conventional Lending Guidelines"

        elif 'pmi' in q_lower or 'mortgage insurance' in q_lower or 'mip' in q_lower:
            return f"🛡️ MORTGAGE INSURANCE:\n\n{FHA_KNOWLEDGE['mortgage_insurance']}\n\n" + \
                   f"Source: FHA/PMI Guidelines"

        elif 'required document' in q_lower or 'what document' in q_lower:
            return f"📋 REQUIRED DOCUMENTS:\n\n{FHA_KNOWLEDGE['required_documents']}\n\n" + \
                   f"Source: FHA Loan Documentation Requirements"

    # Check if asking about completeness
    completeness_keywords = ['missing', 'complete', 'need', 'require',
                            'checklist', 'what do i need', 'what else']

    if any(kw in q_lower for kw in completeness_keywords):
        present, missing = check_fha_completeness(documents)

        result = f"✅ FHA LOAN PACKAGE COMPLETENESS CHECK:\n\n"
        result += f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"

        result += f"📄 DOCUMENTS PRESENT ({len(present)}):\n"
        for item in present:
            result += f"  ✓ {item}\n"

        result += f"\n⚠️ DOCUMENTS MISSING ({len(missing)}):\n"
        for item in missing:
            result += f"  ✗ {item}\n"

        status = "COMPLETE ✅" if len(missing) == 0 else "INCOMPLETE ⚠️"
        result += f"\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        result += f"STATUS: {status}\n"

        if missing:
            result += f"\n📋 NEXT STEPS:\n"
            result += f"Request the following from borrower:\n"
            for i, item in enumerate(missing[:5], 1):
                result += f"  {i}. {item}\n"

        return result

    # Route query using MoRAG
    strategy = route_query(query)

    # Retrieve relevant chunks
    retrieved_chunks = retrieve_relevant_chunks(query, k=5)

    if not retrieved_chunks:
        return f"❌ No relevant information found in uploaded documents.\n\n" + \
               f"Try asking about FHA requirements or upload more documents."

    # Extract precise answer
    answer = extract_precise_data(query, retrieved_chunks, strategy)

    # Format response
    result = f"{answer}\n\n"
    result += f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
    result += f"📊 Strategy: {strategy} | Sources: {retrieved_chunks[0]['doc_type']}"
    if len(retrieved_chunks) > 1:
        result += f", {retrieved_chunks[1]['doc_type']}"

    return result

print("✅ Query handler ready")

✅ Query handler ready


In [ ]:
def process_mortgage_query(pdf_file, user_query):
    """Main processing function for Gradio"""
    global documents

    if not pdf_file:
        return "⚠️ Please upload a PDF file to analyze."

    if not user_query or user_query.strip() == "":
        user_query = "Are there any missing documents for FHA approval?"

    try:
        # Process PDF
        print("📄 Processing PDF...")
        documents = split_pdf_by_document(pdf_file.name)

        # DEBUG: Print what was detected
        print(f"\n🔍 DETECTED {len(documents)} DOCUMENTS:")
        for i, doc in enumerate(documents, 1):
            print(f"  {i}. {doc['type']} (Pages: {doc['pages']})")
            print(f"     Text length: {len(doc['text'])} chars")
            print(f"     First 200 chars: {doc['text'][:200]}")
            print()

        print("🔄 Creating contextual chunks...")
        chunks = create_contextual_chunks(documents)

        print("🔄 Building vector index...")
        num_chunks = build_vector_index(chunks)

        print("✅ Processing complete!")

        # Handle query
        response = handle_query(user_query, documents, chunks)

        return response

    except Exception as e:
        import traceback
        return f"❌ Error processing request: {str(e)}\n\n{traceback.format_exc()}"

# Create Gradio interface
with gr.Blocks(title="Mortgage Document Q&A System", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🏠 Mortgage Document Intelligence System
    ### Contextual Retrieval + MoRAG Adaptive Routing

    **Upload your mortgage loan package and ask questions!**

    This system can:
    - ✅ Answer questions about your specific documents
    - ✅ Check FHA compliance and completeness
    - ✅ Explain FHA rules and requirements
    - ✅ Extract precise data (names, amounts, dates, etc.)
    """)

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(
                label="📁 Upload Mortgage Documents (PDF)",
                file_types=[".pdf"]
            )

            query_input = gr.Textbox(
                label="💬 Ask a Question",
                placeholder="Examples: What is the borrower's name? What is the loan amount? Are there missing documents?",
                lines=3,
                value="What is the borrower's name and loan amount?"
            )

            submit_btn = gr.Button("🔍 Analyze", variant="primary", size="lg")

            gr.Markdown("""
            **Example Questions:**
            - What is the borrower's name?
            - What is the loan amount and interest rate?
            - What is the monthly payment?
            - Are there any missing documents?
            - What are the FHA requirements?
            - What are the closing costs?
            - Is this package FHA compliant?
            """)

        with gr.Column(scale=2):
            output = gr.Textbox(
                label="📊 Response",
                lines=25,
                max_lines=30,
                show_copy_button=True
            )

    submit_btn.click(
        fn=process_mortgage_query,
        inputs=[pdf_input, query_input],
        outputs=output
    )

    gr.Markdown("""
    ---
    **Technical Stack:**
    • Contextual Retrieval (Anthropic, 2024)
    • MoRAG Adaptive Routing (4 strategies: SPARSE, DENSE, FILTERED, HYBRID)
    • FAISS Vector Search
    • Rule-based Precision Extraction
    • Built-in FHA Knowledge Base
    """)

demo.launch(share=True, debug=True)
print("🚀 Mortgage Q&A System Launched!")

/tmp/ipython-input-3157601656.py:42: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Mortgage Document Q&A System", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://38f03f6a7e97213525.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


📄 Processing PDF...
⚠️ Page 10 has minimal text, switching to OCR...
🔄 Using OCR for text extraction...
📸 OCR on page 1...
📸 OCR on page 2...
📸 OCR on page 3...
📸 OCR on page 4...
📸 OCR on page 5...
📸 OCR on page 6...
📸 OCR on page 7...
📸 OCR on page 8...
📸 OCR on page 9...
📸 OCR on page 10...
📸 OCR on page 11...
📄 Page 1: Loan Application (Form 1003) (1099 chars)
📄 Page 2: Closing Disclosure (1296 chars)
📄 Page 3: Other Document (279 chars)
📄 Page 4: Paystub (904 chars)
📄 Page 5: Paystub (904 chars)
📄 Page 6: Paystub (994 chars)
📄 Page 7: Bank Statement (1192 chars)
📄 Page 8: Property Appraisal (1299 chars)
📄 Page 9: Homeowners Insurance (1609 chars)
📄 Page 10: Other Document (1 chars)
📄 Page 11: Other Document (1 chars)

✅ EXTRACTED 8 DOCUMENTS:
  1. Loan Application (Form 1003) (Pages [1], 1100 chars)
  2. Closing Disclosure (Pages [2], 1297 chars)
  3. Other Document (Pages [3], 280 chars)
  4. Paystub (Pages [4, 5, 6], 2805 chars)
  5. Bank Statement (Pages [7], 1193 chars)
  6. P

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔄 Generating embeddings...
🔄 Building FAISS index...
✅ Vector index built: 7 chunks
✅ Processing complete!
